# FunSearch DEC Cloud API 实验（Colab）

本 notebook 目标：在 Colab 上一键跑通云端 API 版 FunSearch + Two-Stage DEC，并输出可用于对比的指标文件。

## 你关心的 4 个点
- 与本地 LLM 版 **同一套去重实现**（Stage1 + Stage2，代码路径一致）。
- 默认已使用当前推荐参数：`stage1=10`、`stage2=128`、`max_non_code_retries=2`。
- 包含详细分步说明（连通性、快速 smoke、正式矩阵、汇总）。
- 包含结果统计展示与指标解释（Time Saved / API Efficiency / To-target / Quality）。


## 0) 运行前准备
1. 在 Colab `Runtime` 中启用 Python 3。
2. 准备好仓库地址和云端 API Key。
3. 如需代理，填写 `HTTPS_PROXY/HTTP_PROXY`。


In [ ]:
# 1) 克隆仓库（修改为你的仓库）
REPO_URL = "https://github.com/ultrabababa/FunSearch-DEC.git"
!git clone $REPO_URL
%cd FunSearch-DEC/funsearch


In [ ]:
# 2) 安装依赖
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install pandas matplotlib


In [ ]:
# 3) 云端 API 配置（必填）
import os

os.environ['FUNSEARCH_CLOUD_API_KEY'] = '<YOUR_API_KEY>'
os.environ['FUNSEARCH_CLOUD_BASE_URL'] = 'https://api.bltcy.ai'
os.environ['FUNSEARCH_CLOUD_MODEL'] = 'gpt-5-nano'

# 可选：代理（按需打开）
# os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7897'
# os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7897'

# 推荐默认参数（已调优）
os.environ['FUNSEARCH_DISABLE_THINKING'] = 'auto'
os.environ['FUNSEARCH_THINKING_PARAM_MODE'] = 'both'
os.environ['FUNSEARCH_MAX_NON_CODE_RETRIES'] = '2'


In [ ]:
# 4) 连通性检查（先确保这个通过）
!python tools/test_cloud_api_config.py --timeout 30


## 5) “仅 OR3 快速 smoke 模式（repeats=1）”按钮单元
- 先用这个快速验证完整链路没问题，再跑全量。


In [ ]:
# 点击运行本单元 = 执行 OR3 smoke (repeats=1)
SMOKE_DATASET = 'OR3'
SMOKE_MAX_SAMPLES = 20
SMOKE_REPEATS = 1

!python tools/run_experiment_matrix.py   --dataset "$SMOKE_DATASET"   --max-samples $SMOKE_MAX_SAMPLES   --repeats $SMOKE_REPEATS   --stage1-case-count 10   --stage2-random-cases 128   --max-non-code-retries 2

!python tools/summarize_experiment_matrix.py --dataset "$SMOKE_DATASET" --repeats $SMOKE_REPEATS


## 6) 正式全量实验（OR3 + Weibull）
- 这一步会自动：
  - 跑矩阵 baseline/dedup
  - 自动从 baseline 中位数推断 target
  - 生成带 target 的最终汇总


In [ ]:
!python tools/run_best_config_pipeline.py   --max-samples 50   --repeats 10   --stage1-case-count 10   --stage2-random-cases 128   --max-non-code-retries 2


In [ ]:
# 7) 读取 aggregate 结果
import json
from pathlib import Path

for name in ['summary_OR3.json', 'summary_Weibull_5k.json']:
    p = Path('logs/experiments') / name
    data = json.loads(p.read_text(encoding='utf-8'))
    print('\n===', name, '===')
    print(json.dumps(data.get('aggregate', {}), indent=2, ensure_ascii=False))


## 8) 指标解释（与你们 proposal 对齐）
- `median_time_saved_ratio`：总 wall-clock 节省比例（>0 更好）
- `median_pipeline_time_saved_ratio`：sample+evaluate 时间节省比例（>0 更好）
- `target_reached_count_baseline/dedup`：达到目标分数的 run 数
- `median_target_time_saved_ratio_both_reached`：仅在两边都达标的 run 中，达标时间节省比例（>0 更好）
- `best_score_diff_dedup_minus_baseline`：质量差（>0 代表 dedup 分数更好）


In [ ]:
# 9) 表格与图（快速可视化）
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

or3 = pd.read_csv(Path('logs/experiments/summary_OR3.csv'))
wb = pd.read_csv(Path('logs/experiments/summary_Weibull_5k.csv'))

display(or3.head())
display(wb.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(or3['repeat'], or3['time_saved_ratio'], marker='o')
axes[0].axhline(0, color='gray', linestyle='--')
axes[0].set_title('OR3 time_saved_ratio')
axes[0].set_xlabel('repeat')

axes[1].plot(wb['repeat'], wb['time_saved_ratio'], marker='o', color='tab:orange')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set_title('Weibull time_saved_ratio')
axes[1].set_xlabel('repeat')

plt.tight_layout()
plt.show()


In [ ]:
# 10) 下载关键结果文件
from google.colab import files

for f in [
    'logs/experiments/summary_OR3.csv',
    'logs/experiments/summary_OR3.json',
    'logs/experiments/summary_Weibull_5k.csv',
    'logs/experiments/summary_Weibull_5k.json',
]:
    files.download(f)
